# Prediction for the same dataset of year 2025 using the datset of those of year 2021 to 2024

In [77]:
# ============================================================
# CPI 2024 PREDICTION USING HISTORICAL TREND
# Training: 2014-2019 + 2022-2023
# COVID years 2020-2021 excluded
# Test/Prediction year: 2024
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



In [78]:
df1= pd.read_csv(r"C:\Users\himan\Education1\Projects\MoSPI_Updated\Data\merged\merged_cpi_rebased_2012.csv")
df1.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2012,2014,January,All India,Combined,Food and Beverages,Cereals and Products,119,10.33
1,2012,2014,January,All India,Combined,Food and Beverages,Meat and Fish,118,10.72
2,2012,2014,January,All India,Combined,Food and Beverages,Egg,124,12.82
3,2012,2014,January,All India,Combined,Food and Beverages,Fruits,113,10.37
4,2012,2014,January,All India,Combined,Food and Beverages,Vegetables,122,19.57


In [79]:
print("\nColumns:")
print(df1.columns.tolist())


Columns:
['BYear', 'Year_i', 'Month_i', 'State_i', 'Sector', 'Group_i', 'SubGroup', 'Index_i', 'Inflation']


In [80]:
# ============================================================
# 4. MAKE SURE YEAR AND CPI ARE NUMERIC
# ============================================================

df1["Year_i"] = pd.to_numeric(
    df1["Year_i"],
    errors="coerce"
)

df1["Index_i"] = pd.to_numeric(
    df1["Index_i"],
    errors="coerce"
)

# Remove rows where year or CPI is missing
df1 = df1.dropna(
    subset=["Year_i", "Index_i"]
)

df1["Year_i"] = df1["Year_i"].astype(int)


In [81]:
# ============================================================
# 5. CLEAN STRING COLUMNS
# ============================================================

string_columns = [
    "Month_i",
    "State_i",
    "Sector",
    "Group_i",
    "SubGroup"
]

for col in string_columns:
    df1[col] = df1[col].astype(str).str.strip()

In [82]:
# ============================================================
# 6. DEFINE THE CPI SERIES
# ============================================================

# Each unique combination represents one CPI series.

keys = [
    "Month_i",
    "State_i",
    "Sector",
    "Group_i",
    "SubGroup"
]

print("\nNumber of unique CPI series:")
print(df1[keys].drop_duplicates().shape[0])


Number of unique CPI series:
29736


In [83]:
# ============================================================
# 7. DEFINE TRAINING AND TEST YEARS
# ============================================================
# 2020 and 2021 = COVID period
#
# Training:
# 2014-2019 and 2022-23
# Test:
# 2024

training_years = [
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2022,
    2023
]

test_year = 2024

train_data = df1[
    df1["Year_i"].isin(training_years)
].copy()

test_data = df1[
    df1["Year_i"] == test_year
].copy()

print("\nTraining years:")
print(training_years)

print("\nTraining rows:", len(train_data))
print("2024 test rows:", len(test_data))



Training years:
[2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023]

Training rows: 163008
2024 test rows: 20376


In [84]:
# ============================================================
# 8. CHECK COVID YEARS
# ============================================================

covid_data = df1[
    df1["Year_i"].isin([2020, 2021])
]

print("\nCOVID rows excluded from training:", len(covid_data))


COVID rows excluded from training: 32577


In [85]:
# ============================================================
# 9. PREDICT 2024
# ============================================================

predictions = []

# Group the training data by the exact CPI series.
#
# January 2014
# January 2015
# ...
# January 2023
# are used to predict January 2024.


grouped_train = train_data.groupby(keys, sort=False)

total_series = len(grouped_train)
processed_series = 0

print("\nStarting 2024 prediction...")
print("Total historical series:", total_series)

for series_key, train_group in grouped_train:

    processed_series += 1

    # --------------------------------------------------------
    # Get corresponding 2024 observation
    # --------------------------------------------------------

    condition = np.ones(len(test_data), dtype=bool)

    for col, value in zip(keys, series_key):
        condition &= (
            test_data[col].values == value
        )

    actual_2024 = test_data.loc[condition]

    # If this series does not exist in 2024,
    # there is nothing to predict.
    if len(actual_2024) == 0:
        continue

    # --------------------------------------------------------
    # Remove missing CPI values
    # --------------------------------------------------------

    train_group = train_group.dropna(
        subset=["Index_i"]
    )

    # We need at least 2 historical observations
    # for linear regression.
    if len(train_group) < 2:
        continue

    # --------------------------------------------------------
    # Prepare X and y
    # --------------------------------------------------------

    X = train_group[["Year_i"]]
    y = train_group["Index_i"]

    # --------------------------------------------------------
    # Linear Regression
    # --------------------------------------------------------

    model = LinearRegression()

    model.fit(X, y)

    # --------------------------------------------------------
    # Predict 2024
    # --------------------------------------------------------

    predicted_2024 = model.predict(
        np.array([[2024]])
    )[0]

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    for _, actual_row in actual_2024.iterrows():

        result = actual_row.to_dict()

        result["Predicted_Index_2024"] = predicted_2024

        # Number of historical observations used
        result["Training_Observations"] = len(train_group)

        # Model slope
        result["Trend_Slope"] = model.coef_[0]

        predictions.append(result)

    # Progress
    if processed_series % 1000 == 0:
        print(
            f"Processed {processed_series}/{total_series} series"
        )



Starting 2024 prediction...
Total historical series: 20376


c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py

Processed 1000/20376 series


c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py

Processed 2000/20376 series


...

c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py

Processed 19000/20376 series


c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py

Processed 20000/20376 series


c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\himan\Education1\Projects\MoSPI_Updated\.venv\Lib\site-packages\sklearn\utils\validation.py

In [86]:
# ============================================================
# 10. CREATE PREDICTION DATAFRAME
# ============================================================

predicted_2024 = pd.DataFrame(predictions)

print("\nPrediction completed!")

print(
    "Predicted 2024 rows:",
    len(predicted_2024)
)


Prediction completed!
Predicted 2024 rows: 20376


In [87]:
# ============================================================
# 11. CHECK PREDICTIONS
# ============================================================

print("\nFirst 10 predictions:")

print(
    predicted_2024[
        keys
        + [
            "Index_i",
            "Predicted_Index_2024",
            "Training_Observations",
            "Trend_Slope"
        ]
    ].head(10)
)




First 10 predictions:
   Month_i    State_i    Sector             Group_i  \
0  January  All India  Combined  Food and Beverages   
1  January  All India  Combined  Food and Beverages   
2  January  All India  Combined  Food and Beverages   
3  January  All India  Combined  Food and Beverages   
4  January  All India  Combined  Food and Beverages   
5  January  All India  Combined  Food and Beverages   
6  January  All India  Combined  Food and Beverages   
7  January  All India  Combined  Food and Beverages   
8  January  All India  Combined  Food and Beverages   
9  January  All India  Combined  Food and Beverages   

                              SubGroup  Index_i  Predicted_Index_2024  \
0                 Cereals and Products      187            167.625000   
1                        Meat and Fish      213            214.166667   
2                                  Egg      205            192.666667   
3                               Fruits      171            162.500000   
4     

In [88]:
# ============================================================
# 12. CALCULATE PREDICTION ERROR
# ============================================================

actual = predicted_2024["Index_i"]

predicted = predicted_2024[
    "Predicted_Index_2024"
]

# Error
predicted_2024["Prediction_Error"] = (
    actual - predicted
)

# Absolute error
predicted_2024["Absolute_Error"] = (
    abs(actual - predicted)
)

# Percentage error
predicted_2024["Percentage_Error"] = (
    abs(actual - predicted) / actual
) * 100

# ============================================================
# 13. EVALUATION METRICS
# ============================================================

MAE = mean_absolute_error(
    actual,
    predicted
)

MSE = mean_squared_error(
    actual,
    predicted
)

RMSE = np.sqrt(MSE)

MAPE = np.mean(
    np.abs(
        (actual - predicted) / actual
    )
) * 100

R2 = r2_score(
    actual,
    predicted
)

# Approximate accuracy
accuracy = 100 - MAPE

# ============================================================
# 14. DISPLAY METRICS
# ============================================================

print("\n" + "=" * 60)
print("2024 CPI PREDICTION EVALUATION")
print("=" * 60)

print(f"MAE  : {MAE:.4f}")
print(f"MSE  : {MSE:.4f}")
print(f"RMSE : {RMSE:.4f}")
print(f"MAPE : {MAPE:.4f}%")
print(f"R²   : {R2:.4f}")

print(
    f"Approximate Accuracy : {accuracy:.4f}%"
)

print("=" * 60)




2024 CPI PREDICTION EVALUATION
MAE  : 9.6718
MSE  : 257.9878
RMSE : 16.0620
MAPE : 4.7102%
R²   : 0.7000
Approximate Accuracy : 95.2898%


In [89]:
# ============================================================
# 15. CREATE METRICS DATAFRAME
# ============================================================

metrics = pd.DataFrame({
    "Metric": [
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Approximate Accuracy"
    ],

    "Value": [
        MAE,
        MSE,
        RMSE,
        MAPE,
        R2,
        accuracy
    ]
})

print("\nEvaluation Metrics:")
print(metrics)




Evaluation Metrics:
                 Metric       Value
0                   MAE    9.671750
1                   MSE  257.987833
2                  RMSE   16.062000
3                  MAPE    4.710172
4                    R2    0.700024
5  Approximate Accuracy   95.289828


In [90]:
# Get actual 2023 data
df1_2023 = df1[df1["Year_i"] == 2023].copy()

# Create lookup using the same keys
index_2023_lookup = (
    df1_2023
    .set_index(keys)["Index_i"]
)

# Match actual 2023 Index with predicted 2024 rows
predicted_2024["Index_2023"] = (
    predicted_2024.set_index(keys).index.map(index_2023_lookup)
)

In [91]:
predicted_2024["Predicted_Inflation_2024"] = (
    (
        predicted_2024["Predicted_Index_2024"]
        - predicted_2024["Index_2023"]
    )
    / predicted_2024["Index_2023"]
) * 100

In [92]:
print(predicted_2024.columns.tolist())

['BYear', 'Year_i', 'Month_i', 'State_i', 'Sector', 'Group_i', 'SubGroup', 'Index_i', 'Inflation', 'Predicted_Index_2024', 'Training_Observations', 'Trend_Slope', 'Prediction_Error', 'Absolute_Error', 'Percentage_Error', 'Index_2023', 'Predicted_Inflation_2024']


In [93]:
df1_2024 = df1[df1["Year_i"] == 2024].copy()

actual_inflation_lookup = (
    df1_2024
    .set_index(keys)["Inflation"]
)

predicted_2024["Actual_Inflation_2024"] = (
    predicted_2024.set_index(keys).index.map(actual_inflation_lookup)
)

In [94]:
predicted_2024.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation,Predicted_Index_2024,Training_Observations,Trend_Slope,Prediction_Error,Absolute_Error,Percentage_Error,Index_2023,Predicted_Inflation_2024,Actual_Inflation_2024
0,2012,2024,January,All India,Combined,Food and Beverages,Cereals and Products,187,7.83,167.625000,8,5.125000,19.375000,19.375000,10.360963,173,-3.106936,7.83
1,2012,2024,January,All India,Combined,Food and Beverages,Meat and Fish,213,1.19,214.166667,8,10.361111,-1.166667,1.166667,0.547731,210,1.984127,1.19
2,2012,2024,January,All India,Combined,Food and Beverages,Egg,205,5.55,192.666667,8,7.902778,12.333333,12.333333,6.016260,194,-0.687285,5.55
3,2012,2024,January,All India,Combined,Food and Beverages,Fruits,171,8.59,162.500000,8,4.500000,8.500000,8.500000,4.970760,158,2.848101,8.59
4,2012,2024,January,All India,Combined,Food and Beverages,Vegetables,195,27.10,165.833333,8,4.222222,29.166667,29.166667,14.957265,153,8.387800,27.10


In [95]:
predicted_2024["Prediction_Error"] = (
    predicted_2024["Predicted_Inflation_2024"]
    - predicted_2024["Actual_Inflation_2024"]
)

predicted_2024["Absolute_Error"] = (
    predicted_2024["Prediction_Error"].abs()
)

In [96]:
print(predicted_2024.head())

   BYear  Year_i  Month_i    State_i    Sector             Group_i  \
0   2012    2024  January  All India  Combined  Food and Beverages   
1   2012    2024  January  All India  Combined  Food and Beverages   
2   2012    2024  January  All India  Combined  Food and Beverages   
3   2012    2024  January  All India  Combined  Food and Beverages   
4   2012    2024  January  All India  Combined  Food and Beverages   

               SubGroup  Index_i  Inflation  Predicted_Index_2024  \
0  Cereals and Products      187       7.83            167.625000   
1         Meat and Fish      213       1.19            214.166667   
2                   Egg      205       5.55            192.666667   
3                Fruits      171       8.59            162.500000   
4            Vegetables      195      27.10            165.833333   

   Training_Observations  Trend_Slope  Prediction_Error  Absolute_Error  \
0                      8     5.125000        -10.936936       10.936936   
1             

In [97]:
evaluation_2024 = predicted_2024.dropna(
    subset=[
        "Predicted_Inflation_2024",
        "Actual_Inflation_2024"
    ]
).copy()

In [98]:
print("Rows available for evaluation:", len(evaluation_2024))

Rows available for evaluation: 20376


In [101]:
accuracy = (
    1 -
    np.sum(np.abs(y_actual - y_pred))
    / np.sum(np.abs(y_actual))
) * 100

print(f"Accuracy : {accuracy:.2f}%")

Accuracy : 8.49%


In [102]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

y_actual = evaluation_2024["Actual_Inflation_2024"]
y_pred = evaluation_2024["Predicted_Inflation_2024"]

mae = mean_absolute_error(y_actual, y_pred)

mse = mean_squared_error(y_actual, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_actual, y_pred)

# Accuracy-like measure
accuracy = (
    1 -
    np.sum(np.abs(y_actual - y_pred))
    / np.sum(np.abs(y_actual))
) * 100

print("2024 Inflation Prediction Performance")
print("--------------------------------------")
print(f"MAE      : {mae:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"R²       : {r2:.4f}")
print(f"Accuracy : {accuracy:.2f}%")

2024 Inflation Prediction Performance
--------------------------------------
MAE      : 5.3481
MSE      : 70.5301
RMSE     : 8.3982
R²       : -0.4843
Accuracy : 8.49%


In [55]:
predicted_2024.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation,Predicted_Index_2024,Training_Observations,Trend_Slope,Prediction_Error,Absolute_Error,Percentage_Error
0,2012,2024,January,All India,Combined,Food and Beverages,Cereals and Products,187,7.83,167.625000,8,5.125000,19.375000,19.375000,10.360963
1,2012,2024,January,All India,Combined,Food and Beverages,Meat and Fish,213,1.19,214.166667,8,10.361111,-1.166667,1.166667,0.547731
2,2012,2024,January,All India,Combined,Food and Beverages,Egg,205,5.55,192.666667,8,7.902778,12.333333,12.333333,6.016260
3,2012,2024,January,All India,Combined,Food and Beverages,Fruits,171,8.59,162.500000,8,4.500000,8.500000,8.500000,4.970760
4,2012,2024,January,All India,Combined,Food and Beverages,Vegetables,195,27.10,165.833333,8,4.222222,29.166667,29.166667,14.957265


In [104]:
actual = predicted_2024["Index_i"]

predicted = predicted_2024["Predicted_Index_2024"]

print(actual)
print(predicted)

0        187
1        213
2        205
3        171
4        195
        ... 
20371    191
20372    190
20373    175
20374    183
20375    185
Name: Index_i, Length: 20376, dtype: int64
0        167.625000
1        214.166667
2        192.666667
3        162.500000
4        165.833333
            ...    
20371    185.166667
20372    189.958333
20373    176.291667
20374    179.500000
20375    182.041667
Name: Predicted_Index_2024, Length: 20376, dtype: float64


In [56]:
predicted_2024.shape

(20376, 15)

In [57]:
predicted_2024.to_csv(r"C:\Users\himan\Education1\Projects\MoSPI_Updated\Data\merged\merged_pred_2024_using_ml.csv", index=False)

# Now Prediction for year 2025 using the same accuracy that have been for 2024